In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import silhouette_score, normalized_mutual_info_score, f1_score, top_k_accuracy_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt

# 设置随机种子确保可复现
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

class ResNetBackbone(nn.Module):
    def __init__(self, base_encoder, projection_dim=128):
        super().__init__()
        try:
            self.backbone = base_encoder(weights=torchvision.models.ResNet50_Weights.DEFAULT)
        except AttributeError:
            self.backbone = base_encoder(pretrained=True)

        self.backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()
        self.backbone.fc = nn.Identity()
        self.feature_dim = 2048

    def forward(self, x):
        return self.backbone(x)

class ProjectionHead(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=2048, output_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim),
            nn.BatchNorm1d(output_dim)
        )

    def forward(self, x):
        return self.mlp(x)

class SimCLRModel(nn.Module):
    def __init__(self, backbone, projection_head):
        super().__init__()
        self.encoder = backbone
        self.projector = projection_head

    def forward(self, x):
        features = self.encoder(x)
        projections = self.projector(features)
        projections = nn.functional.normalize(projections, dim=1)
        return projections, features

class SimCLRTransform:
    def __init__(self, image_size):
        color_jitter = transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
        self.pil_train_transform = transforms.Compose([
            transforms.RandomResizedCrop(size=image_size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([color_jitter], p=0.8),
            transforms.RandomGrayscale(p=0.2),
        ])
        self.to_tensor_normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.507, 0.487, 0.441],
                                std=[0.267, 0.256, 0.276]),
        ])

    def __call__(self, x):
        from torchvision.transforms.functional import to_pil_image
        if isinstance(x, torch.Tensor):
            x = to_pil_image(x)
        v1 = self.pil_train_transform(x)
        v2 = self.pil_train_transform(x)
        return self.to_tensor_normalize(v1), self.to_tensor_normalize(v2)

class CIFAR100WithClassAsCluster(torch.utils.data.Dataset):
    def __init__(self, root, train=True, transform=None, download=True):
        self.base_dataset = datasets.CIFAR100(root=root, train=train, download=download)
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, label = self.base_dataset[idx]
        if self.transform is not None:
            v1, v2 = self.transform(img)
        else:
            v1 = v2 = img
        return v1, v2, label, idx  

# 损失函数
def hard_negative_clustered_simclr_loss(z1, z2, cluster_labels_batch, temperature=0.2, hard_neg_weight=1.5):
    device = z1.device
    batch_size = z1.shape[0]
    z_full = torch.cat((z1, z2), dim=0)
    sim_matrix_full = torch.mm(z_full, z_full.T) / temperature

    pos_mask = torch.zeros((2 * batch_size, 2 * batch_size), dtype=torch.bool, device=device)
    pos_mask[torch.arange(batch_size), torch.arange(batch_size) + batch_size] = True
    pos_mask[torch.arange(batch_size) + batch_size, torch.arange(batch_size)] = True

    mask_neg = torch.ones((2 * batch_size, 2 * batch_size), dtype=torch.bool, device=device)
    mask_neg.fill_diagonal_(False)
    mask_neg[pos_mask] = False

    cl_labels_expanded = cluster_labels_batch.repeat(2)
    same_cluster_mask_full = (cl_labels_expanded.unsqueeze(0) == cl_labels_expanded.unsqueeze(1))
    cluster_neg_mask = ~same_cluster_mask_full
    cluster_neg_mask[pos_mask] = True

    hard_neg_threshold = 0.5
    hard_neg_mask = (sim_matrix_full > hard_neg_threshold) & mask_neg & cluster_neg_mask

    weight_matrix = torch.ones_like(sim_matrix_full, device=device)
    weight_matrix[hard_neg_mask] = hard_neg_weight

    sim_matrix_weighted = sim_matrix_full * weight_matrix
    final_neg_mask = mask_neg & cluster_neg_mask
    sim_matrix_final = sim_matrix_weighted.masked_fill(~final_neg_mask, float('-inf'))

    pos_sim = torch.sum(z1 * z2, dim=1) / temperature
    pos_sim_full = torch.cat([pos_sim, pos_sim], dim=0)

    exp_sim_neg = torch.exp(sim_matrix_final)
    sum_exp_neg = exp_sim_neg.sum(dim=1)
    exp_pos_sim = torch.exp(pos_sim_full)

    log_prob_pos = pos_sim_full - torch.log(exp_pos_sim + sum_exp_neg)
    loss = -log_prob_pos.mean()

    return loss

def get_cosine_scheduler(optimizer, epochs, warmup_epochs=10):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / warmup_epochs
        return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (epochs - warmup_epochs)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def plot_loss_curves(all_losses, save_path='loss_curves.png'):
    plt.figure(figsize=(10,4))
    plt.plot(all_losses)
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

# 线性评估
def evaluate_linear(model, train_loader, test_loader, num_classes=100, epochs=100, device='cuda'):
    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    head = nn.Sequential(
        nn.Linear(model.encoder.feature_dim, 2048),
        nn.BatchNorm1d(2048),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(2048, num_classes)
    ).to(device)

    opt = optim.SGD(head.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    best = 0.0
    for ep in range(epochs):
        head.train()
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)
            with torch.no_grad():
                f = model.encoder(x)
            loss = criterion(head(f), y)
            opt.zero_grad()
            loss.backward()
            opt.step()

        head.eval()
        correct, total = 0,0
        with torch.no_grad():
            for x,y in test_loader:
                x,y = x.to(device), y.to(device)
                pred = head(model.encoder(x)).argmax(1)
                correct += (pred==y).sum().item()
                total += y.size(0)
        acc = 100.*correct/total
        if acc>best: best=acc
        print(f"Linear Epoch {ep+1:2d} | Acc={acc:.2f}% | Best={best:.2f}%")
        scheduler.step()
    return best

# 主函数
def main():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    batch_size = 512
    lr = 1e-3
    temperature = 0.1
    epochs = 100
    image_size = 32

    train_dataset = CIFAR100WithClassAsCluster(
        root='./data', train=True, transform=SimCLRTransform(image_size), download=True
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)

    # 模型
    backbone = ResNetBackbone(torchvision.models.resnet50)
    projector = ProjectionHead()
    model = SimCLRModel(backbone, projector).to(device)

    # 优化
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-6)
    scheduler = get_cosine_scheduler(optimizer, epochs)
    scaler = torch.cuda.amp.GradScaler()

    # 训练
    model.train()
    all_losses = []
    for epoch in range(epochs):
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for v1, v2, cls_label, _ in pbar:
            v1, v2, cls_label = v1.to(device), v2.to(device), cls_label.to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                z1, _ = model(v1)
                z2, _ = model(v2)
                loss = hard_negative_clustered_simclr_loss(z1, z2, cls_label, temperature=temperature)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item())
        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        all_losses.append(avg_loss)
        print(f"Epoch {epoch+1} Avg Loss: {avg_loss:.3f}")

    plot_loss_curves(all_losses)

    # 线性测试
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.507,0.487,0.441],[0.267,0.256,0.276])
    ])
    lt = DataLoader(datasets.CIFAR100('./data',train=True,transform=eval_transform), batch_size=512, shuffle=True, num_workers=2)
    lv = DataLoader(datasets.CIFAR100('./data',train=False,transform=eval_transform), batch_size=512, shuffle=False, num_workers=2)

    best_acc = evaluate_linear(model, lt, lv, epochs=100, device=device)
    print(f"\nFINAL BEST ACCURACY: {best_acc:.2f}%")

if __name__ == "__main__":
    main()

Using device: cuda
Files already downloaded and verified


Epoch 1/100: 100%|██████████| 97/97 [00:42<00:00,  2.30it/s, loss=7.31]


Epoch 1 Avg Loss: 7.249


Epoch 2/100: 100%|██████████| 97/97 [00:41<00:00,  2.32it/s, loss=4.09]


Epoch 2 Avg Loss: 5.303


...

Epoch 98 Avg Loss: 0.587


Epoch 99/100: 100%|██████████| 97/97 [00:41<00:00,  2.32it/s, loss=0.518]


Epoch 99 Avg Loss: 0.584


Epoch 100/100: 100%|██████████| 97/97 [00:41<00:00,  2.34it/s, loss=0.61] 


Epoch 100 Avg Loss: 0.578
Linear Epoch  1 | Acc=67.87% | Best=67.87%
Linear Epoch  2 | Acc=69.51% | Best=69.51%
Linear Epoch  3 | Acc=70.63% | Best=70.63%
Linear Epoch  4 | Acc=71.53% | Best=71.53%
Linear Epoch  5 | Acc=72.11% | Best=72.11%
Linear Epoch  6 | Acc=72.49% | Best=72.49%
Linear Epoch  7 | Acc=72.47% | Best=72.49%
Linear Epoch  8 | Acc=72.83% | Best=72.83%
Linear Epoch  9 | Acc=72.92% | Best=72.92%
Linear Epoch 10 | Acc=72.90% | Best=72.92%
Linear Epoch 11 | Acc=72.99% | Best=72.99%
Linear Epoch 12 | Acc=73.47% | Best=73.47%
Linear Epoch 13 | Acc=73.46% | Best=73.47%
Linear Epoch 14 | Acc=73.26% | Best=73.47%
Linear Epoch 15 | Acc=74.08% | Best=74.08%
Linear Epoch 16 | Acc=73.53% | Best=74.08%
Linear Epoch 17 | Acc=73.39% | Best=74.08%
Linear Epoch 18 | Acc=73.81% | Best=74.08%
Linear Epoch 19 | Acc=73.86% | Best=74.08%
Linear Epoch 20 | Acc=73.69% | Best=74.08%
Linear Epoch 21 | Acc=73.94% | Best=74.08%
Linear Epoch 22 | Acc=74.39% | Best=74.39%
Linear Epoch 23 | Acc=74.63%